# Exercise 3 — MACD Cross

The MACD crossover is a momentum signal. The MACD line (fast EMA − slow EMA) measures how quickly price is accelerating. The signal line smooths the MACD. When MACD crosses above its signal line, momentum is building upward — go long. When it crosses below, momentum has turned negative — go flat.

In [ ]:
import pandas as pd, math, warnings

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def _sma(s, w):  return s.rolling(w).mean()
def _ema(s, w):  return s.ewm(span=w, adjust=False).mean()
def _rsi(s, w):
    d = s.diff()
    g = d.clip(lower=0).rolling(w).mean()
    l = (-d.clip(upper=0)).rolling(w).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    return 100 - (100 / (1 + rs))
def sma_crossover(df, fast=20, slow=50):
    close = df["Close"]
    return (_sma(close, fast) > _sma(close, slow)).fillna(False).astype(int)
def rsi_mean_reversion(df, window=14, oversold=30, overbought=70):
    rsi_s  = _rsi(df["Close"], window)
    signal = pd.Series(float("nan"), index=df.index)
    signal[rsi_s < oversold]   = 1.0
    signal[rsi_s > overbought] = 0.0
    return signal.ffill().fillna(0).astype(int)

def macd_cross(df, fast=12, slow=26, signal=9):
    """Long (1) when the MACD line is above the signal line; flat (0) otherwise.

    Steps:
      1. close       = df["Close"]
      2. macd_line   = _ema(close, fast) - _ema(close, slow)
      3. signal_line = _ema(macd_line, signal)
      4. return (macd_line > signal_line).astype(int)

    No NaN values — EMA starts from the first observation.
    """
    # TODO: implement the 4 steps above
    return pd.Series(0, index=df.index)


### Checks

In [ ]:
checks = 0

# 1 — returns same-length Series, no NaN, values in {0,1}
try:
    df  = _synthetic()
    sig = macd_cross(df)
    assert isinstance(sig, pd.Series) and len(sig) == len(df)
    assert not sig.isna().any()
    assert set(sig.unique()).issubset({0, 1})
    checks += 1; print("✅ 1 valid Series: same length, no NaN, values in {0,1}")
except Exception as e:
    print("❌ 1:", e)

# 2 — produces both 0s and 1s on sine-wave data
try:
    sig = macd_cross(_synthetic())
    assert (sig == 1).any(), "no 1s — MACD never crossed above signal"
    assert (sig == 0).any(), "no 0s — MACD always above signal?"
    checks += 1; print("✅ 2 produces both 0s and 1s")
except Exception as e:
    print("❌ 2:", e)

# 3 — signal matches macd_line > signal_line
try:
    df          = _synthetic()
    close       = df["Close"]
    macd_line   = _ema(close, 12) - _ema(close, 26)
    signal_line = _ema(macd_line, 9)
    expected    = (macd_line > signal_line).astype(int)
    sig         = macd_cross(df, 12, 26, 9)
    assert (sig == expected).all(), "signal doesn't match macd > signal"
    checks += 1; print("✅ 3 signal matches macd_line > signal_line exactly")
except Exception as e:
    print("❌ 3:", e)

# 4 — for a monotonically rising series, MACD > signal most of the time
try:
    dates   = pd.date_range("2023-01-01", periods=100, freq="B")
    rising  = pd.DataFrame({"Close": [float(i) for i in range(1, 101)]}, index=dates)
    sig     = macd_cross(rising, fast=5, slow=20, signal=3)
    # After warmup (last 50 rows), signal should be 1 (fast ema above slow)
    tail_sig = sig.iloc[-50:]
    assert (tail_sig == 1).all(), f"rising prices: expected all 1s in tail, got {tail_sig.value_counts().to_dict()}"
    checks += 1; print("✅ 4 monotonically rising prices → MACD signal = 1 at tail")
except Exception as e:
    print("❌ 4:", e)

# 5 — for a monotonically falling series, MACD < signal at tail
try:
    dates    = pd.date_range("2023-01-01", periods=100, freq="B")
    falling  = pd.DataFrame({"Close": [float(100 - i) for i in range(100)]}, index=dates)
    sig      = macd_cross(falling, fast=5, slow=20, signal=3)
    tail_sig = sig.iloc[-50:]
    assert (tail_sig == 0).all(), f"falling prices: expected all 0s in tail, got {tail_sig.value_counts().to_dict()}"
    checks += 1; print("✅ 5 monotonically falling prices → MACD signal = 0 at tail")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
